In [1]:
# %pip install mteb==2.10.12 accelerate>=1.1.0 sentence-transformers==5.3.0 dotenv torch==2.10.0+cu128 numpy==2.0.2 datasets==4.8.3 requests==2.32.4 scikit-learn==1.6.1 scipy==1.16.3 polars==1.35.2

In [2]:
import mteb
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
import torch
from dotenv import load_dotenv

load_dotenv()

/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
tasks = mteb.get_tasks(languages=["por"], modalities=['text'])

for task in tasks:
    print(task)

BibleNLPBitextMining(name='BibleNLPBitextMining', languages=['eng', 'por'])
FloresBitextMining(name='FloresBitextMining', languages=['ace', 'acm', 'acq', '...'])
NTREXBitextMining(name='NTREXBitextMining', languages=['arb', 'ben', 'cat', '...'])
TatoebaBitextMining(name='Tatoeba', languages=['eng', 'por'])
WebFAQBitextMiningQAs(name='WebFAQBitextMiningQAs', languages=['cat', 'dan', 'deu', '...'])
WebFAQBitextMiningQuestions(name='WebFAQBitextMiningQuestions', languages=['cat', 'dan', 'deu', '...'])
AfriSentiClassification(name='AfriSentiClassification', languages=['por'])
AfriSentiLangClassification(name='AfriSentiLangClassification', languages=['amh', 'arq', 'ary', '...'])
LanguageClassification(name='LanguageClassification', languages=['ara', 'bul', 'cmn', '...'])
MassiveIntentClassification(name='MassiveIntentClassification', languages=['por'])
MassiveScenarioClassification(name='MassiveScenarioClassification', languages=['por'])
MultiHateClassification(name='MultiHateClassification

In [5]:
model_list = ["iara-project/NeoBERTugues-simcse-pt-ckpt-1000"]
dims_list = [None, 512, 256, 128, 64]
# None, 512, 256, 128, 64
filepath = '../data/results_eval_mteb_finetuned.csv'

tasks = [
    ("Assin2STS", None),
    ("SICK-BR-STS", None),
    ("STSBenchmarkMultilingualSTS", 'pt'),
    
    ('MassiveIntentClassification', 'pt'),
    ('MultiHateClassification', 'por'),
    ('BrazilianToxicTweetsClassification', None),
    ('HateSpeechPortugueseClassification', None),
    ('TweetSentimentClassification', 'portuguese'),

    ('MultiLongDocReranking', 'pt'),
    ('WikipediaRerankingMultilingual', 'pt'),
    ('XGlueWPRReranking', 'pt'),

    ('WebFAQRetrieval', 'por'),
    ('MultiLongDocRetrieval', 'pt'),
    ('WikipediaRetrievalMultilingual', 'pt')
]

In [ ]:
for model_name in model_list:
    model_meta = SentenceTransformer(model_name, device=DEVICE)
    for truncate_dim in dims_list:
        # select the desired tasks and evaluate
        task_name_list = []
        model_name_list = []
        main_score_list = []
        truncate_dims_list = []
        for task_info in tasks:
            print(f"""
#############################

[{model_name} - {truncate_dim} dims] Avaliando {task_info[0]} ({task_info[1]})...

#############################
            """)

            task = mteb.get_task(task_info[0], languages=['por'], hf_subsets=task_info[1])

            # with encode kwargs
            result = mteb.evaluate(model_meta, task, encode_kwargs={"batch_size": 32, "truncate_dim": truncate_dim}, cache=None)

            task_name = result.task_results[0].task_name
            model_name = result.model_name
            main_score = result.task_results[0].main_score

            task_name_list.append(task_name)
            model_name_list.append(model_name)
            main_score_list.append(main_score)
            truncate_dims_list.append(truncate_dim)

            print(f"Main Score: {main_score}")

            del task, result
            torch.cuda.empty_cache()
        
        df_results = pd.DataFrame({
            'model_name': model_name_list,
            'embedding_dim': truncate_dims_list,
            'task_name': task_name_list,
            'main_score': main_score_list
        })

        if os.path.exists(filepath):
            df_results_cache = pd.read_csv(filepath)
            df_results = pd.concat([df_results_cache, df_results], axis=0, ignore_index=True)

        df_results.to_csv(filepath, index=False)

        print(f"Avaliação concluída para {model_name} - {truncate_dim} dims!")

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 5556.25it/s]



#############################

[iara-project/NeoBERTugues-simcse-pt-ckpt-1000 - None dims] Avaliando Assin2STS (None)...

#############################
            
Main Score: 0.45095991321953816

#############################

[iara-project/NeoBERTugues-simcse-pt-ckpt-1000 - None dims] Avaliando SICK-BR-STS (None)...

#############################
            
Main Score: 0.45284755831108153

#############################

[iara-project/NeoBERTugues-simcse-pt-ckpt-1000 - None dims] Avaliando STSBenchmarkMultilingualSTS (pt)...

#############################
            
Main Score: 0.46141109862739704

#############################

[iara-project/NeoBERTugues-simcse-pt-ckpt-1000 - None dims] Avaliando MassiveIntentClassification (pt)...

#############################
            


/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control thi